# 📊 Phân tích Luật kết hợp (Association Rules) theo Phân cụm Khách hàng

Notebook này thực hiện:
1. Đọc file phân cụm khách hàng (`customer_segmentation.csv`) và dữ liệu giao dịch
2. Chia dữ liệu giao dịch thành 3 bộ riêng cho từng cụm
3. Thống kê cơ bản (shape) cho từng bộ dữ liệu

In [1]:
# === Import Libraries ===
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries loaded!')

✅ Libraries loaded!


## 1. 📥 Đọc dữ liệu phân cụm và dữ liệu giao dịch

In [2]:
# --- 1.1 Đọc file phân cụm khách hàng ---
df_segmentation = pd.read_csv('customer_segmentation.csv')
print(f'📦 customer_segmentation.csv')
print(f'   Shape: {df_segmentation.shape[0]:,} dòng × {df_segmentation.shape[1]} cột')
print(f'   Columns: {list(df_segmentation.columns)}')
print(f'   Số cụm: {df_segmentation["cluster"].nunique()}')
print(f'   Phân bố cụm:')
print(df_segmentation['cluster'].value_counts().sort_index())
print()
df_segmentation.head()

📦 customer_segmentation.csv
   Shape: 206,209 dòng × 20 cột
   Columns: ['user_id', 'total_orders', 'average_days_between_orders', 'reorder_rate', 'avg_basket_size', 'unique_products', 'product_diversity', 'weekend_ratio', 'night_order_ratio', 'organic_ratio_order', 'cluster', 'total_orders_scaled', 'average_days_between_orders_scaled', 'reorder_rate_scaled', 'avg_basket_size_scaled', 'unique_products_scaled', 'product_diversity_scaled', 'weekend_ratio_scaled', 'night_order_ratio_scaled', 'organic_ratio_order_scaled']
   Số cụm: 3
   Phân bố cụm:
cluster
0    113906
1     40204
2     52099
Name: count, dtype: int64



,user_id,total_orders,average_days_between_orders,reorder_rate,avg_basket_size,unique_products,product_diversity,weekend_ratio,night_order_ratio,organic_ratio_order,cluster,total_orders_scaled,average_days_between_orders_scaled,reorder_rate_scaled,avg_basket_size_scaled,unique_products_scaled,product_diversity_scaled,weekend_ratio_scaled,night_order_ratio_scaled,organic_ratio_order_scaled
0,1,10,20.259259,0.694915,5.900000,18,0.305085,0.00,0.0,0.254237,0,0.189189,0.675309,0.702269,0.214078,0.096317,0.297731,0.00,0.0,0.275213
1,2,14,15.967033,0.476923,13.928571,102,0.523077,0.00,0.0,0.256410,0,0.297297,0.532234,0.481970,0.564840,0.572238,0.518030,0.00,0.0,0.277566
2,3,12,11.487179,0.625000,7.333333,33,0.375000,0.50,0.0,0.340909,0,0.243243,0.382906,0.631614,0.276699,0.181303,0.368386,0.50,0.0,0.369036
3,4,5,15.357143,0.055556,3.600000,17,0.944444,0.20,0.0,0.111111,0,0.054054,0.511905,0.056143,0.113592,0.090652,0.943857,0.20,0.0,0.120278
4,5,4,14.500000,0.378378,9.250000,23,0.621622,0.25,0.0,0.486486,0,0.027027,0.483333,0.382382,0.360437,0.124646,0.617618,0.25,0.0,0.526625


In [3]:
# --- 1.2 Đọc dữ liệu đơn hàng (orders) ---
df_orders = pd.read_csv('Data/orders.csv')
print(f'📦 orders.csv')
print(f'   Shape: {df_orders.shape[0]:,} dòng × {df_orders.shape[1]} cột')
print(f'   Columns: {list(df_orders.columns)}')
print()
df_orders.head()

📦 orders.csv
   Shape: 3,421,083 dòng × 7 cột
   Columns: ['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']



,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


In [4]:
# --- 1.3 Đọc dữ liệu chi tiết giao dịch (order_products__prior) ---
df_order_products = pd.read_csv('Data/order_products__prior.csv')
print(f'📦 order_products__prior.csv')
print(f'   Shape: {df_order_products.shape[0]:,} dòng × {df_order_products.shape[1]} cột')
print(f'   Columns: {list(df_order_products.columns)}')
print()
df_order_products.head()

📦 order_products__prior.csv
   Shape: 32,434,489 dòng × 4 cột
   Columns: ['order_id', 'product_id', 'add_to_cart_order', 'reordered']



,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


In [5]:
# --- 1.4 Đọc danh sách sản phẩm ---
df_products = pd.read_csv('Data/products.csv')
print(f'📦 products.csv')
print(f'   Shape: {df_products.shape[0]:,} dòng × {df_products.shape[1]} cột')
print(f'   Columns: {list(df_products.columns)}')
print()
df_products.head()

📦 products.csv
   Shape: 49,688 dòng × 4 cột
   Columns: ['product_id', 'product_name', 'aisle_id', 'department_id']



,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


## 2. 🔗 Kết hợp dữ liệu giao dịch với thông tin cụm

Bước này ghép thông tin cluster vào dữ liệu giao dịch thông qua:
- `orders.csv` → chứa `user_id` và `order_id`
- `customer_segmentation.csv` → chứa `user_id` và `cluster`
- `order_products__prior.csv` → chứa `order_id` và `product_id`

In [6]:
# --- 2.1 Lấy cột user_id và cluster từ file phân cụm ---
df_user_cluster = df_segmentation[['user_id', 'cluster']].copy()
print(f'📌 Bảng user_id - cluster: {df_user_cluster.shape}')
print()

# --- 2.2 Lọc chỉ lấy các đơn hàng eval_set='prior' (dữ liệu lịch sử) ---
df_orders_prior = df_orders[df_orders['eval_set'] == 'prior'][['order_id', 'user_id']].copy()
print(f'📌 Đơn hàng prior: {df_orders_prior.shape}')
print()

# --- 2.3 Ghép cluster vào đơn hàng thông qua user_id ---
df_orders_with_cluster = df_orders_prior.merge(df_user_cluster, on='user_id', how='inner')
print(f'📌 Đơn hàng + cluster: {df_orders_with_cluster.shape}')
print()

# --- 2.4 Ghép thông tin sản phẩm vào giao dịch ---
df_transactions = df_order_products.merge(df_orders_with_cluster, on='order_id', how='inner')
print(f'📌 Giao dịch đầy đủ (order_id, product_id, user_id, cluster): {df_transactions.shape}')
print()

# --- 2.5 Ghép thêm tên sản phẩm ---
df_transactions = df_transactions.merge(df_products[['product_id', 'product_name']], on='product_id', how='left')
print(f'📌 Giao dịch + tên sản phẩm: {df_transactions.shape}')
print()
df_transactions.head(10)

📌 Bảng user_id - cluster: (206209, 2)

📌 Đơn hàng prior: (3214874, 2)

📌 Đơn hàng + cluster: (3214874, 3)

📌 Giao dịch đầy đủ (order_id, product_id, user_id, cluster): (32434489, 6)

📌 Giao dịch + tên sản phẩm: (32434489, 7)



,order_id,product_id,add_to_cart_order,reordered,user_id,cluster,product_name
0,2,33120,1,1,202279,0,Organic Egg Whites
1,2,28985,2,1,202279,0,Michigan Organic Kale
2,2,9327,3,0,202279,0,Garlic Powder
3,2,45918,4,1,202279,0,Coconut Butter
4,2,30035,5,0,202279,0,Natural Sweetener
5,2,17794,6,1,202279,0,Carrots
6,2,40141,7,1,202279,0,Original Unflavored Gelatine Mix
7,2,1819,8,1,202279,0,All Natural No Stir Creamy Almond Butter
8,2,43668,9,0,202279,0,Classic Blend Cole Slaw
9,3,33754,1,1,205970,2,Total 2% with Strawberry Lowfat Greek Strained...


## 3. ✂️ Chia dữ liệu giao dịch theo từng cụm

Tách bộ dữ liệu giao dịch thành **3 bộ riêng biệt**, mỗi bộ ứng với 1 cụm khách hàng (cluster 0, 1, 2).

In [7]:
# --- 3.1 Chia giao dịch theo cluster ---
df_cluster_0 = df_transactions[df_transactions['cluster'] == 0].copy()
df_cluster_1 = df_transactions[df_transactions['cluster'] == 1].copy()
df_cluster_2 = df_transactions[df_transactions['cluster'] == 2].copy()

print('=' * 70)
print('✂️ KẾT QUẢ CHIA DỮ LIỆU GIAO DỊCH THEO CỤM')
print('=' * 70)
print()

for i, df_cluster in enumerate([df_cluster_0, df_cluster_1, df_cluster_2]):
    n_users = df_cluster['user_id'].nunique()
    n_orders = df_cluster['order_id'].nunique()
    n_products = df_cluster['product_id'].nunique()
    print(f'📊 Cluster {i}:')
    print(f'   Shape: {df_cluster.shape[0]:,} dòng × {df_cluster.shape[1]} cột')
    print(f'   Số khách hàng (user_id): {n_users:,}')
    print(f'   Số đơn hàng (order_id):  {n_orders:,}')
    print(f'   Số sản phẩm (product_id): {n_products:,}')
    print(f'   Trung bình sản phẩm/đơn hàng: {df_cluster.shape[0] / n_orders:.1f}')
    print()

✂️ KẾT QUẢ CHIA DỮ LIỆU GIAO DỊCH THEO CỤM

📊 Cluster 0:
   Shape: 7,509,390 dòng × 7 cột
   Số khách hàng (user_id): 113,906
   Số đơn hàng (order_id):  838,264
   Số sản phẩm (product_id): 48,271
   Trung bình sản phẩm/đơn hàng: 9.0

📊 Cluster 1:
   Shape: 4,099,627 dòng × 7 cột
   Số khách hàng (user_id): 40,204
   Số đơn hàng (order_id):  442,594
   Số sản phẩm (product_id): 45,006
   Trung bình sản phẩm/đơn hàng: 9.3

📊 Cluster 2:
   Shape: 20,825,472 dòng × 7 cột
   Số khách hàng (user_id): 52,099
   Số đơn hàng (order_id):  1,934,016
   Số sản phẩm (product_id): 48,741
   Trung bình sản phẩm/đơn hàng: 10.8



In [8]:
# --- 3.2 Xem mẫu dữ liệu từng cluster ---

print('=' * 70)
print('🔍 MẪU DỮ LIỆU CLUSTER 0')
print('=' * 70)
display(df_cluster_0.head())
print()

print('=' * 70)
print('🔍 MẪU DỮ LIỆU CLUSTER 1')
print('=' * 70)
display(df_cluster_1.head())
print()

print('=' * 70)
print('🔍 MẪU DỮ LIỆU CLUSTER 2')
print('=' * 70)
display(df_cluster_2.head())

🔍 MẪU DỮ LIỆU CLUSTER 0


,order_id,product_id,add_to_cart_order,reordered,user_id,cluster,product_name
0,2,33120,1,1,202279,0,Organic Egg Whites
1,2,28985,2,1,202279,0,Michigan Organic Kale
2,2,9327,3,0,202279,0,Garlic Powder
3,2,45918,4,1,202279,0,Coconut Butter
4,2,30035,5,0,202279,0,Natural Sweetener



🔍 MẪU DỮ LIỆU CLUSTER 1


,order_id,product_id,add_to_cart_order,reordered,user_id,cluster,product_name
59,7,34050,1,0,142903,1,Orange Juice
60,7,46802,2,0,142903,1,Pineapple Chunks
241,27,13176,1,1,129389,1,Bag of Organic Bananas
242,27,30442,2,1,129389,1,Low Fat Vanilla Yogurt
243,27,13646,3,1,129389,1,Lemon Hummus



🔍 MẪU DỮ LIỆU CLUSTER 2


,order_id,product_id,add_to_cart_order,reordered,user_id,cluster,product_name
9,3,33754,1,1,205970,2,Total 2% with Strawberry Lowfat Greek Strained...
10,3,24838,2,1,205970,2,Unsweetened Almondmilk
11,3,17704,3,1,205970,2,Lemons
12,3,21903,4,1,205970,2,Organic Baby Spinach
13,3,17668,5,1,205970,2,Unsweetened Chocolate Almond Breeze Almond Milk


In [9]:
# --- 3.3 Tổng hợp so sánh giữa các cluster ---

summary = pd.DataFrame({
    'Cluster': [0, 1, 2],
    'Số giao dịch (dòng)': [len(df_cluster_0), len(df_cluster_1), len(df_cluster_2)],
    'Số khách hàng': [
        df_cluster_0['user_id'].nunique(),
        df_cluster_1['user_id'].nunique(),
        df_cluster_2['user_id'].nunique()
    ],
    'Số đơn hàng': [
        df_cluster_0['order_id'].nunique(),
        df_cluster_1['order_id'].nunique(),
        df_cluster_2['order_id'].nunique()
    ],
    'Số sản phẩm': [
        df_cluster_0['product_id'].nunique(),
        df_cluster_1['product_id'].nunique(),
        df_cluster_2['product_id'].nunique()
    ]
})

summary['TB sản phẩm/đơn'] = (summary['Số giao dịch (dòng)'] / summary['Số đơn hàng']).round(1)
summary['TB đơn/khách'] = (summary['Số đơn hàng'] / summary['Số khách hàng']).round(1)

print('=' * 70)
print('📋 BẢNG TỔNG HỢP SO SÁNH CÁC CLUSTER')
print('=' * 70)
display(summary)

📋 BẢNG TỔNG HỢP SO SÁNH CÁC CLUSTER


,Cluster,Số giao dịch (dòng),Số khách hàng,Số đơn hàng,Số sản phẩm,TB sản phẩm/đơn,TB đơn/khách
0,0,7509390,113906,838264,48271,9.0,7.4
1,1,4099627,40204,442594,45006,9.3,11.0
2,2,20825472,52099,1934016,48741,10.8,37.1


## 4. 🧹 Tóm gọn 3 bộ dữ liệu cho Luật Kết Hợp (Association Rules)

Để chạy luật kết hợp (Apriori), mỗi bộ dữ liệu chỉ cần:
- **`order_id`**: mã đơn hàng (đại diện cho 1 giao dịch/giỏ hàng)
- **`product_name`**: tên sản phẩm trong giỏ hàng đó

Loại bỏ các cột không cần thiết (`product_id`, `add_to_cart_order`, `reordered`, `user_id`, `cluster`).

In [10]:
# --- 4.1 Tóm gọn: chỉ giữ order_id và product_name ---
# Toàn bộ dữ liệu tổng hợp (Không giới hạn/chia cụm)
basket_all = df_transactions[['order_id', 'product_name']].copy()

basket_cluster_0 = df_cluster_0[['order_id', 'product_name']].copy()
basket_cluster_1 = df_cluster_1[['order_id', 'product_name']].copy()
basket_cluster_2 = df_cluster_2[['order_id', 'product_name']].copy()

# Hiển thị minh họa: Gom các sản phẩm thành từng giỏ hàng (Basket/Transaction) thực sự
print('=' * 70)
print('🧹 DỮ LIỆU TÓM GỌN CHO LUẬT KẾT HỢP')
print('=' * 70)
print()

for name, basket in zip(['ALL (Giới hạn gốc)', 'Cluster 0', 'Cluster 1', 'Cluster 2'], [basket_all, basket_cluster_0, basket_cluster_1, basket_cluster_2]):
    n_orders = basket['order_id'].nunique()
    n_products = basket['product_name'].nunique()
    print(f'📦 {name}:')
    print(f'   Shape: {basket.shape[0]:,} dòng × {basket.shape[1]} cột')
    print(f'   Số giỏ hàng (đơn hàng): {n_orders:,}')
    print(f'   Số sản phẩm duy nhất:   {n_products:,}')
    print()

print('🔍 VÍ DỤ MINH HỌA: Cách dữ liệu được gom thành từng Giỏ Hàng (Basket)')
# Lấy 3 order ngẫu nhiên làm ví dụ từ cluster 0
sample_orders = basket_cluster_0['order_id'].unique()[:3]
for oid in sample_orders:
    items = basket_cluster_0[basket_cluster_0['order_id'] == oid]['product_name'].tolist()
    print(f'  🛒 Đơn hàng #{oid}: {items}')


🧹 DỮ LIỆU TÓM GỌN CHO LUẬT KẾT HỢP

📦 ALL (Giới hạn gốc):
   Shape: 32,434,489 dòng × 2 cột
   Số giỏ hàng (đơn hàng): 3,214,874
   Số sản phẩm duy nhất:   49,677

📦 Cluster 0:
   Shape: 7,509,390 dòng × 2 cột
   Số giỏ hàng (đơn hàng): 838,264
   Số sản phẩm duy nhất:   48,271

📦 Cluster 1:
   Shape: 4,099,627 dòng × 2 cột
   Số giỏ hàng (đơn hàng): 442,594
   Số sản phẩm duy nhất:   45,006

📦 Cluster 2:
   Shape: 20,825,472 dòng × 2 cột
   Số giỏ hàng (đơn hàng): 1,934,016
   Số sản phẩm duy nhất:   48,741

🔍 VÍ DỤ MINH HỌA: Cách dữ liệu được gom thành từng Giỏ Hàng (Basket)
  🛒 Đơn hàng #2: ['Organic Egg Whites', 'Michigan Organic Kale', 'Garlic Powder', 'Coconut Butter', 'Natural Sweetener', 'Carrots', 'Original Unflavored Gelatine Mix', 'All Natural No Stir Creamy Almond Butter', 'Classic Blend Cole Slaw']
  🛒 Đơn hàng #6: ['Cleanse', 'Dryer Sheets Geranium Scent', 'Clean Day Lavender Scent Room Freshener Spray']
  🛒 Đơn hàng #11: ['Teriyaki & Pineapple Chicken Meatballs', 'Mango 

In [11]:
# --- 4.2 Xem mẫu dữ liệu tóm gọn ---
print('📦 Mẫu basket_cluster_0:')
display(basket_cluster_0.head(10))
print()
print('📦 Mẫu basket_cluster_1:')
display(basket_cluster_1.head(10))
print()
print('📦 Mẫu basket_cluster_2:')
display(basket_cluster_2.head(10))

📦 Mẫu basket_cluster_0:


,order_id,product_name
0,2,Organic Egg Whites
1,2,Michigan Organic Kale
2,2,Garlic Powder
3,2,Coconut Butter
4,2,Natural Sweetener
5,2,Carrots
6,2,Original Unflavored Gelatine Mix
7,2,All Natural No Stir Creamy Almond Butter
8,2,Classic Blend Cole Slaw
56,6,Cleanse



📦 Mẫu basket_cluster_1:


,order_id,product_name
59,7,Orange Juice
60,7,Pineapple Chunks
241,27,Bag of Organic Bananas
242,27,Low Fat Vanilla Yogurt
243,27,Lemon Hummus
244,27,Roasted Red Pepper Hummus
245,27,Dark Chocolate Chips
246,27,Organic Low Sodium Vegetable Broth
247,27,Organic Avocado
248,27,Carrots



📦 Mẫu basket_cluster_2:


,order_id,product_name
9,3,Total 2% with Strawberry Lowfat Greek Strained...
10,3,Unsweetened Almondmilk
11,3,Lemons
12,3,Organic Baby Spinach
13,3,Unsweetened Chocolate Almond Breeze Almond Milk
14,3,Organic Ginger Root
15,3,Air Chilled Organic Boneless Skinless Chicken ...
16,3,Organic Ezekiel 49 Bread Cinnamon Raisin
17,4,Plain Pre-Sliced Bagels
18,4,Honey/Lemon Cough Drops


## 5. 🚀 Triển khai Logical AND Apriori (Từ tài liệu chiến lược tối ưu)

Dựa trên báo cáo nghiên cứu: *"Optimization of Association Rule Mining on Large Datasets"*.
Thuật toán tối ưu Apriori bằng cách:
1. Biểu diễn mỗi sản phẩm bằng một **Bit Vector** (Mỗi bit đại diện cho 1 giao dịch: Bit = 1 nếu có mua sản phẩm, 0 nếu không).
2. Tính độ hỗ trợ (Support) tần suất khách mua chung 2 sản phẩm bằng phép tính **Toán tử Logic AND** trên 2 Bit Vector của chúng.
3. Đếm số lượng bit `1` trong kết quả (thông qua hàm `.bit_count()`) để kiểm tra `min_support`.

Phương pháp này giúp tính toán cực nhanh, tốn ít bộ nhớ và không cần thực hiện nhiều vòng lặp duyệt qua CSDL!

In [12]:
def logical_and_apriori(basket_df, min_support_ratio=0.001, min_confidence=0.1):
    """Triển khai thuật toán Logical AND Apriori"""
    import time
    import pandas as pd
    from IPython.display import display
    
    start_time = time.time()
    
    n_transactions = basket_df['order_id'].nunique()
    min_support_count = int(min_support_ratio * n_transactions)
    
    print(f"⚡ Bắt đầu Logical Apriori... Giới hạn MIN_SUP: {min_support_count} ({min_support_ratio*100:.2f}%)/{n_transactions}")
    
    unique_orders = basket_df['order_id'].unique()
    order_mapping = {order_id: i for i, order_id in enumerate(unique_orders)}
    
    item_bit_vectors = {}
    item_support = {}
    grouped = basket_df.groupby('product_name')
    
    # =========================================================================
    # BƯỚC 2: IN RA BẢNG MẪU LOGICAL TABLE (BINARY) THEO LÝ THUYẾT CHIẾN LƯỢC
    # =========================================================================
    print("\n🔍 2. Chuyển sang Logical Table (binary)")
    print("Mô phỏng dữ liệu giao dịch thành ma trận nhị phân (1=Có mua / 0=Không mua):")
    
    logical_table_sample = pd.DataFrame([
        {'TID': 'T1', 'I1': 1, 'I2': 1, 'I3': 1, 'I4': 1, 'I5': 1, 'I6': 0, 'I7': 0},
        {'TID': 'T2', 'I1': 1, 'I2': 1, 'I3': 0, 'I4': 0, 'I5': 0, 'I6': 0, 'I7': 0},
        {'TID': 'T3', 'I1': 0, 'I2': 0, 'I3': 0, 'I4': 1, 'I5': 1, 'I6': 1, 'I7': 0},
        {'TID': 'T4', 'I1': 1, 'I2': 1, 'I3': 0, 'I4': 0, 'I5': 0, 'I6': 0, 'I7': 1},
        {'TID': 'T5', 'I1': 1, 'I2': 0, 'I3': 1, 'I4': 1, 'I5': 0, 'I6': 0, 'I7': 0}
    ]).set_index('TID')
    
    # In ra với căn giữa chuẩn format
    display(logical_table_sample.style.set_properties(**{'text-align': 'center'}))
    print("... (Dữ liệu thực tế hàng triệu dòng đang được mã hoá bằng Bit Vector bên dưới)\n")
    # =========================================================================
    
    # Thực thi thuật toán thực sự (Nén thành integer Bit Vector)
    for product, group in grouped:
        bit_vector = 0
        for order_id in group['order_id']:
            bit_vector |= (1 << order_mapping[order_id])  # Shift bit và Logical OR
            
        count = bit_vector.bit_count()  # Đếm bit 1 (chính là Support count)
            
        if count >= min_support_count:
            item_bit_vectors[product] = bit_vector
            item_support[product] = count
            
    # 3. Tìm 2-itemsets bằng phép Logical AND
    l2_itemsets = {}
    rules = []
    
    items = list(item_bit_vectors.keys())
    n_items = len(items)
    print(f"Đang phân tích tổ hợp từ {n_items} sản phẩm (Dùng phép toán bitwise AND: &)...")
    
    for i in range(n_items):
        for j in range(i+1, n_items):
            item_a = items[i]
            item_b = items[j]
            
            # Cốt lõi của thuật toán: Tính toán & Cùng lúc
            combined_bit_vector = item_bit_vectors[item_a] & item_bit_vectors[item_b]
            support_both = combined_bit_vector.bit_count()
            
            # Prune L2 và tạo Luật Kết hợp (Rule Formulation)
            if support_both >= min_support_count:
                conf_A_B = support_both / item_support[item_a]
                if conf_A_B >= min_confidence:
                    rules.append({
                        'Rule': f"{item_a} => {item_b}",
                        'Antecedents': item_a,
                        'Consequents': item_b,
                        'Support': support_both / n_transactions,
                        'Confidence': conf_A_B,
                        'Lift': conf_A_B / (item_support[item_b] / n_transactions)
                    })
                    
                conf_B_A = support_both / item_support[item_b]
                if conf_B_A >= min_confidence:
                    rules.append({
                        'Rule': f"{item_b} => {item_a}",
                        'Antecedents': item_b,
                        'Consequents': item_a,
                        'Support': support_both / n_transactions,
                        'Confidence': conf_B_A,
                        'Lift': conf_B_A / (item_support[item_a] / n_transactions)
                    })
    
    end_time = time.time()
    
    df_rules = pd.DataFrame(rules)
    if not df_rules.empty:
        df_rules = df_rules.sort_values(by=['Lift', 'Confidence'], ascending=[False, False])
        
    print(f"\n✅ Hoàn tất! Tìm thấy {len(df_rules)} luật trong vòng {end_time - start_time:.2f} giây.")
    return df_rules



In [13]:
basket_cluster_1

,order_id,product_name
59,7,Orange Juice
60,7,Pineapple Chunks
241,27,Bag of Organic Bananas
242,27,Low Fat Vanilla Yogurt
243,27,Lemon Hummus
...,...,...
32434351,3421061,Flat Fillets of Anchovies
32434352,3421061,Gluten Free Pancake Mix
32434353,3421061,Cauliflower
32434354,3421061,"Olives, Organic, Kalamata, Pitted"


In [14]:
# --- 5.1 Chạy thử thuật toán cho Cluster 1 ---
# Vì dataset tương đối lớn (hơn 400k đơn hàng), ta set:
# - Min Support = 1% (Mua chung xuất hiện ở ít nhất 1% hóa đơn, tức ~4000+ hóa đơn)
# - Min Conf = 10%
print("Bắt đầu phân tích Cluster 1...")
df_rules_c1 = logical_and_apriori(basket_cluster_1, min_support_ratio=0.0003, min_confidence=0.1)

# Hiển thị Top 15 luật mạnh nhất (theo Lift)
if not df_rules_c1.empty:
    display(df_rules_c1.head(15))
else:
    print("Không tìm thấy luật nào đạt yêu cầu min_support và min_confidence. Bạn hãy giảm chúng xuống.")

Bắt đầu phân tích Cluster 1...
⚡ Bắt đầu Logical Apriori... Giới hạn MIN_SUP: 132 (0.03%)/442594

🔍 2. Chuyển sang Logical Table (binary)
Mô phỏng dữ liệu giao dịch thành ma trận nhị phân (1=Có mua / 0=Không mua):


,I1,I2,I3,I4,I5,I6,I7
TID,,,,,,,
T1,1,1,1,1,1,0,0
T2,1,1,0,0,0,0,0
T3,0,0,0,1,1,1,0
T4,1,1,0,0,0,0,1
T5,1,0,1,1,0,0,0


... (Dữ liệu thực tế hàng triệu dòng đang được mã hoá bằng Bit Vector bên dưới)

Đang phân tích tổ hợp từ 5039 sản phẩm (Dùng phép toán bitwise AND: &)...

✅ Hoàn tất! Tìm thấy 3887 luật trong vòng 166.30 giây.


,Rule,Antecedents,Consequents,Support,Confidence,Lift
2550,Oh My Yog! Organic Wild Quebec Blueberry Cream...,Oh My Yog! Organic Wild Quebec Blueberry Cream...,Oh My Yog! Pacific Coast Strawberry Trilayer Y...,0.000474,0.617647,727.039581
2551,Oh My Yog! Pacific Coast Strawberry Trilayer Y...,Oh My Yog! Pacific Coast Strawberry Trilayer Y...,Oh My Yog! Organic Wild Quebec Blueberry Cream...,0.000474,0.558511,727.039581
2544,Oh My Yog! Gingered Pear Trilayer Yogurt => Oh...,Oh My Yog! Gingered Pear Trilayer Yogurt,Oh My Yog! Pacific Coast Strawberry Trilayer Y...,0.000352,0.498403,586.675481
2545,Oh My Yog! Pacific Coast Strawberry Trilayer Y...,Oh My Yog! Pacific Coast Strawberry Trilayer Y...,Oh My Yog! Gingered Pear Trilayer Yogurt,0.000352,0.414894,586.675481
1929,Dairy Free Coconut Milk Blueberry Yogurt Alter...,Dairy Free Coconut Milk Blueberry Yogurt Alter...,Dairy Free Coconut Milk Raspberry Yogurt Alter...,0.000325,0.416185,582.914466
1930,Dairy Free Coconut Milk Raspberry Yogurt Alter...,Dairy Free Coconut Milk Raspberry Yogurt Alter...,Dairy Free Coconut Milk Blueberry Yogurt Alter...,0.000325,0.455696,582.914466
2542,Oh My Yog! Gingered Pear Trilayer Yogurt => Oh...,Oh My Yog! Gingered Pear Trilayer Yogurt,Oh My Yog! Organic Wild Quebec Blueberry Cream...,0.000312,0.440895,573.933208
2543,Oh My Yog! Organic Wild Quebec Blueberry Cream...,Oh My Yog! Organic Wild Quebec Blueberry Cream...,Oh My Yog! Gingered Pear Trilayer Yogurt,0.000312,0.405882,573.933208
2549,Oh My Yog! Pacific Coast Strawberry Trilayer Y...,Oh My Yog! Pacific Coast Strawberry Trilayer Y...,Oh My Yog! Madagascar Vanilla Trilayer Yogyurt,0.000404,0.476064,571.010826
2548,Oh My Yog! Madagascar Vanilla Trilayer Yogyurt...,Oh My Yog! Madagascar Vanilla Trilayer Yogyurt,Oh My Yog! Pacific Coast Strawberry Trilayer Y...,0.000404,0.485095,571.010826


In [15]:
# --- 5.2 Chạy thử thuật toán cho Cluster 0 ---
# Cluster 0 có số lượng khoảng 800k đơn hàng
print("Bắt đầu phân tích Cluster 0...")
df_rules_c0 = logical_and_apriori(basket_cluster_0, min_support_ratio=0.0003, min_confidence=0.1)

if not df_rules_c0.empty:
    print("\n🏆 TRÍCH XUẤT LUẬT CLUSTER 0 (TOP 15 THEO LIFT):")
    display(df_rules_c0.head(15))
else:
    print("Không tìm thấy luật nào đạt yêu cầu min_support và min_confidence. Bạn hãy giảm chúng xuống.")

Bắt đầu phân tích Cluster 0...
⚡ Bắt đầu Logical Apriori... Giới hạn MIN_SUP: 251 (0.03%)/838264

🔍 2. Chuyển sang Logical Table (binary)
Mô phỏng dữ liệu giao dịch thành ma trận nhị phân (1=Có mua / 0=Không mua):


,I1,I2,I3,I4,I5,I6,I7
TID,,,,,,,
T1,1,1,1,1,1,0,0
T2,1,1,0,0,0,0,0
T3,0,0,0,1,1,1,0
T4,1,1,0,0,0,0,1
T5,1,0,1,1,0,0,0


... (Dữ liệu thực tế hàng triệu dòng đang được mã hoá bằng Bit Vector bên dưới)

Đang phân tích tổ hợp từ 5029 sản phẩm (Dùng phép toán bitwise AND: &)...

✅ Hoàn tất! Tìm thấy 3130 luật trong vòng 436.93 giây.

🏆 TRÍCH XUẤT LUẬT CLUSTER 0 (TOP 15 THEO LIFT):


,Rule,Antecedents,Consequents,Support,Confidence,Lift
2117,Oh My Yog! Organic Wild Quebec Blueberry Cream...,Oh My Yog! Organic Wild Quebec Blueberry Cream...,Oh My Yog! Pacific Coast Strawberry Trilayer Y...,0.000433,0.699422,904.784343
2118,Oh My Yog! Pacific Coast Strawberry Trilayer Y...,Oh My Yog! Pacific Coast Strawberry Trilayer Y...,Oh My Yog! Organic Wild Quebec Blueberry Cream...,0.000433,0.560185,904.784343
2114,Oh My Yog! Organic Wild Quebec Blueberry Cream...,Oh My Yog! Organic Wild Quebec Blueberry Cream...,Oh My Yog! Madagascar Vanilla Trilayer Yogyurt,0.000317,0.512524,690.724259
2113,Oh My Yog! Madagascar Vanilla Trilayer Yogyurt...,Oh My Yog! Madagascar Vanilla Trilayer Yogyurt,Oh My Yog! Organic Wild Quebec Blueberry Cream...,0.000317,0.427653,690.724259
2115,Oh My Yog! Madagascar Vanilla Trilayer Yogyurt...,Oh My Yog! Madagascar Vanilla Trilayer Yogyurt,Oh My Yog! Pacific Coast Strawberry Trilayer Y...,0.000366,0.493569,638.489560
2116,Oh My Yog! Pacific Coast Strawberry Trilayer Y...,Oh My Yog! Pacific Coast Strawberry Trilayer Y...,Oh My Yog! Madagascar Vanilla Trilayer Yogyurt,0.000366,0.473765,638.489560
69,Almond Milk Blueberry Yogurt => Almond Milk Pe...,Almond Milk Blueberry Yogurt,Almond Milk Peach Yogurt,0.000452,0.500000,531.219265
70,Almond Milk Peach Yogurt => Almond Milk Bluebe...,Almond Milk Peach Yogurt,Almond Milk Blueberry Yogurt,0.000452,0.480355,531.219265
71,Almond Milk Blueberry Yogurt => Almond Milk St...,Almond Milk Blueberry Yogurt,Almond Milk Strawberry Yogurt,0.000472,0.522427,505.695284
72,Almond Milk Strawberry Yogurt => Almond Milk B...,Almond Milk Strawberry Yogurt,Almond Milk Blueberry Yogurt,0.000472,0.457275,505.695284


In [16]:
# --- 5.3 Chạy thử thuật toán cho Cluster 2 ---
# Cluster 2 lớn nhất (khoảng gần 2 triệu đơn hàng)
print("Bắt đầu phân tích Cluster 2...")
df_rules_c2 = logical_and_apriori(basket_cluster_2, min_support_ratio=0.0003, min_confidence=0.1)

if not df_rules_c2.empty:
    print("\n🏆 TRÍCH XUẤT LUẬT CLUSTER 2 (TOP 15 THEO LIFT):")
    display(df_rules_c2.head(15))
else:
    print("Không tìm thấy luật nào đạt yêu cầu min_support và min_confidence. Bạn hãy giảm chúng xuống.")

Bắt đầu phân tích Cluster 2...
⚡ Bắt đầu Logical Apriori... Giới hạn MIN_SUP: 580 (0.03%)/1934016

🔍 2. Chuyển sang Logical Table (binary)
Mô phỏng dữ liệu giao dịch thành ma trận nhị phân (1=Có mua / 0=Không mua):


,I1,I2,I3,I4,I5,I6,I7
TID,,,,,,,
T1,1,1,1,1,1,0,0
T2,1,1,0,0,0,0,0
T3,0,0,0,1,1,1,0
T4,1,1,0,0,0,0,1
T5,1,0,1,1,0,0,0


... (Dữ liệu thực tế hàng triệu dòng đang được mã hoá bằng Bit Vector bên dưới)

Đang phân tích tổ hợp từ 5279 sản phẩm (Dùng phép toán bitwise AND: &)...

✅ Hoàn tất! Tìm thấy 6773 luật trong vòng 1088.23 giây.

🏆 TRÍCH XUẤT LUẬT CLUSTER 2 (TOP 15 THEO LIFT):


,Rule,Antecedents,Consequents,Support,Confidence,Lift
4225,Oh My Yog! Organic Wild Quebec Blueberry Cream...,Oh My Yog! Organic Wild Quebec Blueberry Cream...,Oh My Yog! Pacific Coast Strawberry Trilayer Y...,0.000480,0.657466,693.699130
4226,Oh My Yog! Pacific Coast Strawberry Trilayer Y...,Oh My Yog! Pacific Coast Strawberry Trilayer Y...,Oh My Yog! Organic Wild Quebec Blueberry Cream...,0.000480,0.506819,693.699130
6757,Unsweetened Whole Milk Mixed Berry Greek Yogur...,Unsweetened Whole Milk Mixed Berry Greek Yogurt,Unsweetened Whole Milk Peach Greek Yogurt,0.000310,0.494641,675.119515
6758,Unsweetened Whole Milk Peach Greek Yogurt => U...,Unsweetened Whole Milk Peach Greek Yogurt,Unsweetened Whole Milk Mixed Berry Greek Yogurt,0.000310,0.423430,675.119515
6760,Unsweetened Whole Milk Strawberry Yogurt => Un...,Unsweetened Whole Milk Strawberry Yogurt,Unsweetened Whole Milk Peach Greek Yogurt,0.000372,0.492803,672.610760
6759,Unsweetened Whole Milk Peach Greek Yogurt => U...,Unsweetened Whole Milk Peach Greek Yogurt,Unsweetened Whole Milk Strawberry Yogurt,0.000372,0.507410,672.610760
4078,"Mighty 4 Sweet Potato, Blueberry, Millet & Gre...","Mighty 4 Sweet Potato, Blueberry, Millet & Gre...","Mighty 4 Kale, Strawberry, Amaranth & Greek Yo...",0.000482,0.462568,502.874785
4077,"Mighty 4 Kale, Strawberry, Amaranth & Greek Yo...","Mighty 4 Kale, Strawberry, Amaranth & Greek Yo...","Mighty 4 Sweet Potato, Blueberry, Millet & Gre...",0.000482,0.524452,502.874785
4223,Oh My Yog! Madagascar Vanilla Trilayer Yogyurt...,Oh My Yog! Madagascar Vanilla Trilayer Yogyurt,Oh My Yog! Pacific Coast Strawberry Trilayer Y...,0.000383,0.468948,494.791601
4224,Oh My Yog! Pacific Coast Strawberry Trilayer Y...,Oh My Yog! Pacific Coast Strawberry Trilayer Y...,Oh My Yog! Madagascar Vanilla Trilayer Yogyurt,0.000383,0.403710,494.791601


In [19]:
# --- 5.4 Chạy thuật toán cho TOÀN BỘ dữ liệu tổng (Dataset Gốc) ---
# Tổng số giao dịch là hơn 3.2 triệu đơn hàng.
print("Bắt đầu phân tích TOÀN BỘ DATASET...")
df_rules_all = logical_and_apriori(basket_all, min_support_ratio=0.0005, min_confidence=0.1)

if not df_rules_all.empty:
    print("\n🏆 TRÍCH XUẤT LUẬT TRÊN TOÀN BỘ DATASET (TOP 15 THEO LIFT):")
    display(df_rules_all.head(15))
else:
    print("Không tìm thấy luật nào.")

Bắt đầu phân tích TOÀN BỘ DATASET...
⚡ Bắt đầu Logical Apriori... Giới hạn MIN_SUP: 1607 (0.05%)/3214874

🔍 2. Chuyển sang Logical Table (binary)
Mô phỏng dữ liệu giao dịch thành ma trận nhị phân (1=Có mua / 0=Không mua):


,I1,I2,I3,I4,I5,I6,I7
TID,,,,,,,
T1,1,1,1,1,1,0,0
T2,1,1,0,0,0,0,0
T3,0,0,0,1,1,1,0
T4,1,1,0,0,0,0,1
T5,1,0,1,1,0,0,0


... (Dữ liệu thực tế hàng triệu dòng đang được mã hoá bằng Bit Vector bên dưới)

Đang phân tích tổ hợp từ 3402 sản phẩm (Dùng phép toán bitwise AND: &)...

✅ Hoàn tất! Tìm thấy 3020 luật trong vòng 1383.54 giây.

🏆 TRÍCH XUẤT LUẬT TRÊN TOÀN BỘ DATASET (TOP 15 THEO LIFT):


,Rule,Antecedents,Consequents,Support,Confidence,Lift
62,Almond Milk Blueberry Yogurt => Almond Milk Pe...,Almond Milk Blueberry Yogurt,Almond Milk Peach Yogurt,0.000703,0.479220,327.306327
63,Almond Milk Peach Yogurt => Almond Milk Bluebe...,Almond Milk Peach Yogurt,Almond Milk Blueberry Yogurt,0.000703,0.480136,327.306327
64,Almond Milk Blueberry Yogurt => Almond Milk St...,Almond Milk Blueberry Yogurt,Almond Milk Strawberry Yogurt,0.000840,0.572943,322.073808
65,Almond Milk Strawberry Yogurt => Almond Milk B...,Almond Milk Strawberry Yogurt,Almond Milk Blueberry Yogurt,0.000840,0.472460,322.073808
66,Almond Milk Peach Yogurt => Almond Milk Strawb...,Almond Milk Peach Yogurt,Almond Milk Strawberry Yogurt,0.000770,0.526238,295.818730
67,Almond Milk Strawberry Yogurt => Almond Milk P...,Almond Milk Strawberry Yogurt,Almond Milk Peach Yogurt,0.000770,0.433118,295.818730
1344,Chocolate Peanut Butter => Coconut Chia Bar,Chocolate Peanut Butter,Coconut Chia Bar,0.000582,0.401546,277.199911
1345,Coconut Chia Bar => Chocolate Peanut Butter,Coconut Chia Bar,Chocolate Peanut Butter,0.000582,0.401546,277.199911
2928,Yotoddler Organic Pear Spinach Mango Yogurt =>...,Yotoddler Organic Pear Spinach Mango Yogurt,Organic Whole Milk Strawberry Beet Berry Yogur...,0.000878,0.460898,235.906522
2927,Organic Whole Milk Strawberry Beet Berry Yogur...,Organic Whole Milk Strawberry Beet Berry Yogur...,Yotoddler Organic Pear Spinach Mango Yogurt,0.000878,0.449451,235.906522


## 6. 💾 Xuất kết quả Luật Kết Hợp ra file CSV

Lưu lại kết quả quá trình phân tích `Association Rules` theo từng cụm khách hàng để phục vụ cho báo cáo hoặc hệ thống Recommender System.

In [20]:
import os

# Tạo thư mục Output nếu chưa có
output_dir = "Output"
os.makedirs(output_dir, exist_ok=True)
try:
    if not df_rules_all.empty:
        path_all = os.path.join(output_dir, 'Association_Rules_ALL_Dataset.csv')
        df_rules_all.to_csv(path_all, index=False, encoding='utf-8-sig')
        print(f'✅ Đã xuất {len(df_rules_all)} luật của TOÀN DATASET ra: {path_all}')
except NameError:
    pass


try:
    # Xuất cụm 0
    if not df_rules_c0.empty:
        path_c0 = os.path.join(output_dir, "Association_Rules_Cluster_0.csv")
        df_rules_c0.to_csv(path_c0, index=False, encoding="utf-8-sig")
        print(f"✅ Đã xuất {len(df_rules_c0)} luật của Cluster 0 ra: {path_c0}")
except NameError:
    pass

try:
    # Xuất cụm 1
    if not df_rules_c1.empty:
        path_c1 = os.path.join(output_dir, "Association_Rules_Cluster_1.csv")
        df_rules_c1.to_csv(path_c1, index=False, encoding="utf-8-sig")
        print(f"✅ Đã xuất {len(df_rules_c1)} luật của Cluster 1 ra: {path_c1}")
except NameError:
    pass

try:
    # Xuất cụm 2
    if not df_rules_c2.empty:
        path_c2 = os.path.join(output_dir, "Association_Rules_Cluster_2.csv")
        df_rules_c2.to_csv(path_c2, index=False, encoding="utf-8-sig")
        print(f"✅ Đã xuất {len(df_rules_c2)} luật của Cluster 2 ra: {path_c2}")
except NameError:
    pass

print("\n🎉 XUẤT DỮ LIỆU THÀNH CÔNG!")

✅ Đã xuất 3020 luật của TOÀN DATASET ra: Output\Association_Rules_ALL_Dataset.csv
✅ Đã xuất 3130 luật của Cluster 0 ra: Output\Association_Rules_Cluster_0.csv
✅ Đã xuất 3887 luật của Cluster 1 ra: Output\Association_Rules_Cluster_1.csv
✅ Đã xuất 6773 luật của Cluster 2 ra: Output\Association_Rules_Cluster_2.csv

🎉 XUẤT DỮ LIỆU THÀNH CÔNG!
